# Chapter 6 — Document Loading

Hands-On LangChain. Code targets **LangChain 1.x** (loaders live in `langchain_community`).

To build an application that chats with your data, you first have to **load** that data into a standard
form. LangChain's document loaders take many formats and sources — PDFs, CSVs, spreadsheets, Word files,
web pages, YouTube — and turn each into the same `Document` object: `page_content` plus `metadata`.

Companion notebook for Chapter 6. The sample files live in `../data/`.

```bash
pip install langchain-community pypdf "unstructured[xlsx]" docx2txt beautifulsoup4 -q
```

## 6.1 Loading PDFs

`PyPDFLoader` loads a PDF into one `Document` per page.

In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/career-in-ai.pdf")
pages = loader.load()
len(pages)

2

In [2]:
page = pages[0]
print(page.page_content[:300])

Building a Career in AI - Field Notes
An original summary of widely-known guidance on starting a career in artificial intelligence, written for this
book's examples.
Who shaped this advice
Much of the popular guidance on entering AI comes from Andrew Ng, the founder of DeepLearning.AI and a
long-tim


Every document carries metadata — here, the source file and page number:

In [3]:
page.metadata

{'producer': 'PyPDF',
 'creator': 'PyPDF',
 'creationdate': '2026-06-08T13:16:59+00:00',
 'source': '../data/career-in-ai.pdf',
 'total_pages': 2,
 'page': 0,
 'page_label': '1'}

## 6.2 Loading CSV files

`CSVLoader` creates one `Document` per row; each field becomes a `key: value` line in the content.

In [4]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path="../data/earthquakes.csv")
data = loader.load()
print(len(data), "documents")
data[:3]

16 documents


[Document(metadata={'source': '../data/earthquakes.csv', 'row': 0}, page_content='date: 1900-01-01\nYear: 1900\nearthquakes_per_year: 13'),
 Document(metadata={'source': '../data/earthquakes.csv', 'row': 1}, page_content='date: 1901-01-01\nYear: 1901\nearthquakes_per_year: 14'),
 Document(metadata={'source': '../data/earthquakes.csv', 'row': 2}, page_content='date: 1902-01-01\nYear: 1902\nearthquakes_per_year: 8')]

## 6.3 Loading Excel files (.xls / .xlsx)

`UnstructuredExcelLoader` reads Microsoft Excel files. In `"elements"` mode it also stores an HTML
representation of the sheet in the document metadata under `text_as_html`.

In [5]:
from langchain_community.document_loaders import UnstructuredExcelLoader

loader = UnstructuredExcelLoader("../data/sample-people.xlsx", mode="elements")
docs = loader.load()
print(docs[0].page_content[:200])

First Name Last Name Gender Country Age Dulce Abril Female United States 32 Mara Hashimoto Female Great Britain 25 Philip Gent Male France 36 Kathleen Hanner Female United States 25 Nereida Magwood Fe


## 6.4 Loading Word files (.doc / .docx)

`Docx2txtLoader` extracts the text from a Word document.

In [6]:
from langchain_community.document_loaders import Docx2txtLoader

loader = Docx2txtLoader("../data/sample.docx")
data = loader.load()
print(data[0].page_content)

Sample Document

This is a sample Word document used to demonstrate the Word loader. LangChain reads the text content of each paragraph into a Document object.

Document loaders convert many file formats into a single standard shape: page_content plus metadata. That uniformity is what makes the rest of a retrieval pipeline format-agnostic.


## 6.5 Loading YouTube videos into text

You can transcribe a YouTube video's audio to text. One approach downloads the audio and runs it
through OpenAI's Whisper model:

```python
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import OpenAIWhisperParser
from langchain_community.document_loaders.blob_loaders.youtube_audio import YoutubeAudioLoader

url = "https://www.youtube.com/watch?v=5p248yoa3oE"   # Andrew Ng: Opportunities in AI
loader = GenericLoader(YoutubeAudioLoader([url], "docs/youtube/"), OpenAIWhisperParser())
docs = loader.load()
docs[0].page_content[:500]
```

This example is **not executed in this notebook**: it downloads a large audio file (tens of MB),
requires `yt-dlp` and `ffmpeg`, and incurs Whisper API cost. It is shown so you have the pattern.

> A lighter, network-only alternative — when a video already has captions — is `YoutubeLoader`, which
> fetches the existing transcript instead of downloading and transcribing audio:
>
> ```python
> from langchain_community.document_loaders import YoutubeLoader
> docs = YoutubeLoader.from_youtube_url(url, add_video_info=False).load()
> ```

## 6.6 Loading HTML pages

For an HTML file on disk, `BSHTMLLoader` (built on BeautifulSoup) extracts the text.

In [7]:
from langchain_community.document_loaders import BSHTMLLoader

loader = BSHTMLLoader("../data/sample-page.html")
docs = loader.load()
print(docs[0].page_content.strip())
print("\nmetadata:", docs[0].metadata)

Getting Started with Generative AI

Getting Started with Generative AI
This page lists a few resources for learning to build LLM applications.
Prompt Engineering
Prompt Engineering Best Practices for Instruction-Tuned LLMs
Iterative Prompt Development
Projects
Building a PDF-Chat App using LangChain and Streamlit
Top 10 Open Source LLMs to Use in Your Next Application

metadata: {'source': '../data/sample-page.html', 'title': 'Getting Started with Generative AI'}


To load a page from the live web instead of disk, use `WebBaseLoader` with a URL (it requires
network access and the `beautifulsoup4` package):

```python
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://example.com")
docs = loader.load()
```

## 6.7 A practical tip: route by file extension

When an application must handle several file types, keep a mapping from extension to loader class, then
pick the right loader at runtime based on the file's extension.

In [8]:
import os
from langchain_community.document_loaders import (
    CSVLoader, UnstructuredExcelLoader, Docx2txtLoader, PyPDFLoader)

DOCUMENT_MAP = {
    ".pdf":  PyPDFLoader,
    ".csv":  CSVLoader,
    ".xls":  UnstructuredExcelLoader,
    ".xlsx": UnstructuredExcelLoader,
    ".docx": Docx2txtLoader,
    ".doc":  Docx2txtLoader,
}

def load_any(path):
    ext = os.path.splitext(path)[1].lower()
    loader_class = DOCUMENT_MAP.get(ext)
    if loader_class is None:
        raise ValueError(f"No loader registered for {ext}")
    return loader_class(path).load()

# the same function handles every registered type
for path in ["../data/sample.docx", "../data/earthquakes.csv"]:
    docs = load_any(path)
    print(f"{path:30} -> {len(docs)} document(s)")

../data/sample.docx            -> 1 document(s)
../data/earthquakes.csv        -> 16 document(s)
